In [1]:
!pip install -q transformers sentence-transformers faiss-cpu
!pip install -q python-docx

In [2]:
# Cell 2: Imports
import torch
from sentence_transformers import SentenceTransformer
from transformers import T5Tokenizer, T5ForConditionalGeneration
import faiss
from google.colab import files
from docx import Document

print("Libraries imported")

Libraries imported


In [3]:
# Cell 3: Load the language model (Flan-T5)
print("Loading language model...")
MODEL_NAME = "google/flan-t5-large"

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

if not torch.cuda.is_available():
    print("WARNING: Running on CPU. This will be slow!")
    print("Enable GPU: Runtime > Change runtime type > T4 GPU")
    model = model.to('cpu')

print(f"Model loaded: {MODEL_NAME}")
print(f"Device: {model.device if torch.cuda.is_available() else 'cpu'}")

Loading language model...


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded: google/flan-t5-large
Device: cuda:0


In [5]:
# Cell 4: Load the embedding model
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded


In [6]:
# Cell 5: Upload and process DOCX
print("Please upload your DOCX file:")
uploaded = files.upload()
docx_path = list(uploaded.keys())[0]

# Extract text from DOCX
def extract_text_from_docx(docx_path):
    doc = Document(docx_path)
    text = ""
    for paragraph in doc.paragraphs:
        text += paragraph.text + "\n"
    return text

doc_text = extract_text_from_docx(docx_path)
print(f"\nExtracted {len(doc_text)} characters from DOCX")
print(f"Preview: {doc_text[:200]}...")

Please upload your DOCX file:


Saving Romanovs.docx to Romanovs (3).docx

Extracted 4888 characters from DOCX
Preview: The Romanov Dynasty of Russia

Russia witnessed the Revolution in the month of February 1917 (old date in the Julian calendar). This revolution is known as the February 1917 Revolution in Russia. The ...


In [7]:
# Cell 6: Chunk the document
def chunk_text(text, chunk_size=100, overlap=20):
    """ Split text into overlapping chunks """

    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)

    return chunks

chunks = chunk_text(doc_text, chunk_size=100, overlap=20)
print(f"Created {len(chunks)} chunks")
print(f"Average chunk length: {sum(len(c.split()) for c in chunks) // len(chunks)} words")
print(f"Preview: {chunks[0][:200]}...")


Created 11 chunks
Average chunk length: 89 words
Preview: The Romanov Dynasty of Russia Russia witnessed the Revolution in the month of February 1917 (old date in the Julian calendar). This revolution is known as the February 1917 Revolution in Russia. The F...


In [8]:
# Cell 7: Create FAISS vector store
print("Creating embeddings and FAISS index...")
embeddings = embedder.encode(chunks, show_progress_bar=True)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))
print(f"FAISS index created with {len(chunks)} vectors")

Creating embeddings and FAISS index...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index created with 11 vectors


In [14]:
# Cell 8: Answer generation for T5 models

def generate_answer(prompt, max_tokens=150):
    """Generate an answer from the T5 model for the given prompt."""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            num_beams = 2,
        )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer.strip()


def retrieve_context(question, k=5):
    """Retrieve top-k most relevant chunks"""

    q_embedding = embedder.encode([question])
    distances, indices = index.search(q_embedding.astype('float32'), k)

    retrieved_chunks = [chunks[i] for i in indices[0]]
    citations = [f"CHUNK {i+1}" for i in indices[0]]

    return retrieved_chunks, citations

def answer_question(question):
    """
    Retrieve relevant chunks and generate an answer grounded in them.
    If the answer isn't in context, the prompt tells the model to say "I don't know"
    """

    # Retrieve relevant context
    retrieved_chunks, citations = retrieve_context(question, k=5)
    context = "\n\n".join(retrieved_chunks)

    prompt = f"""Answer the question based only on the following context. If the answer is not in the context, say "I don't know".

Context: {context}

Question: {question}
Answer:"""

    # Generate answer
    answer = generate_answer(prompt, max_tokens=100)

    # Clean up (Flan-T5 often starts its answers in lowercase)
    answer = answer.strip()
    if answer and answer[0].islower():
        answer = answer[0].upper() + answer[1:]

    # Check for refusal
    if "don't know" in answer.lower():
        return {
            "answer": "I don't know based on the document.",
            "citations": []
        }

    return {
        "answer": answer,
        "citations": citations
    }

print("T5 question-answering functions defined")

T5 question-answering functions defined


In [15]:
# Cell 9: Test specific/set questions
def test_questions():

    test_qs = [
        "What is a boyar?",
        "Who was Elizabeth's father?",
        "When did the Romanov Dynasty rule Russia?",
        "Who was the first Romanov Tsar?",
        "What happened to Nicholas II?",
        "Who was Ivan the Terrible?",
        "What is the capital of France?"  # Not in document - should refuse
    ]

    print("\n" + "="*70)
    print("Running Test Suite")
    print("="*70 + "\n")

    for q in test_qs:
        print(f"Q: {q}")
        result = answer_question(q)
        print(f"A: {result['answer']}")
        print()

test_questions()


Running Test Suite

Q: What is a boyar?
A: They were very high-ranking feudals next to the aristocratic or ruling family

Q: Who was Elizabeth's father?
A: Peter the Great

Q: When did the Romanov Dynasty rule Russia?
A: 1613 AD

Q: Who was the first Romanov Tsar?
A: Michael I

Q: What happened to Nicholas II?
A: Abdicated

Q: Who was Ivan the Terrible?
A: Ivan IV

Q: What is the capital of France?
A: I don't know based on the document.



In [ ]:
# Cell 10: Interactive chat interface
def chat():
    """Interactive question-answering loop. Type 'exit' to quit."""

    print("\n" + "="*70)
    print("Document QA")
    print("="*70)
    print("Type your questions (or 'exit' to quit)")
    print("\nTips for best results:")
    print("  • Ask specific questions about information in the document")
    print("  • Avoid very broad or general questions")
    print("  • The agent will say 'I don't know' if unsure\n")

    while True:
        q = input("You: ").strip()
        if q.lower() in ['exit', 'quit', 'q']:
            print("\nGoodbye!")
            break

        if not q:
            continue

        print("\n[Thinking...]")
        result = answer_question(q)

        print(f"\nAssistant: {result['answer']}")

        if result["citations"]:
            print("\nSources used:")
            for c in result["citations"]:
                print(f"  • {c}")
        print()

# Start the chat
chat()


Document QA
Type your questions (or 'exit' to quit)

Tips for best results:
  • Ask specific questions about information in the document
  • Avoid very broad or general questions
  • The agent will say 'I don't know' if unsure

You: Give me the name of a female tsar

[Thinking...]

Assistant: Anastasiya Romanova Zakharina Yureva

Sources used:
  • CHUNK 3
  • CHUNK 4
  • CHUNK 5
  • CHUNK 7
  • CHUNK 2

You: Do you like mango?

[Thinking...]

Assistant: I don't know based on the document.

